Инструмент для автоматического извлечения данных о медицинских изделиях, соответствующих требованиям FDA 510(k).

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import pypdf
import io
import re
import time
import logging

# 屏蔽 pypdf 内部的字体宽度或格式不标准的警告，保持输出整洁
logging.getLogger("pypdf").setLevel(logging.ERROR)

# 1. 配置区域
INPUT_FILE = 'data.xlsx'          # 你的原始文件名
OUTPUT_FILE = 'data_with_predicates.xlsx' # 处理后的结果文件名
COLUMN_NAME = 'Submission Number' # Excel 中包含 K 编号的列名
SLEEP_TIME = 2                    # 每次请求间隔(秒)，FDA 对高频爬虫较敏感

Основная функция: сканирование и анализ предикатных устройств. Введите K-номер, перейдите на страницу FDA, найдите ссылку на PDF-файл, загрузите и проанализируйте PDF-файл, извлеките предикатное устройство с помощью регулярных выражений и верните очищенный результат.

In [2]:
# 2. 核心功能函数
def fetch_predicate_device(k_number):
    """
    爬取并解析 FDA Summary PDF 以寻找 Predicate Device
    """
    # 基础校验
    if pd.isna(k_number) or str(k_number).strip().lower() in ['', 'nan', 'none']:
        return "N/A"
    
    current_k = str(k_number).strip().upper()
    url = f"https://www.accessdata.fda.gov/scripts/cdrh/cfdocs/cfPMN/pmn.cfm?ID={current_k}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
    }

    try:
        # 第一步：获取网页 HTML
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # 第二步：查找指向 Summary PDF 的链接
        pdf_link = None
        for a_tag in soup.find_all('a', href=True):
            # 匹配包含 'Summary' 文字或者以 .pdf 结尾的链接
            if 'summary' in a_tag.text.lower() or a_tag['href'].lower().endswith('.pdf'):
                href = a_tag['href']
                pdf_link = href if href.startswith('http') else 'https://www.accessdata.fda.gov' + href
                break
        
        if not pdf_link:
            return "Link Not Found"

        # 第三步：下载并解析 PDF
        pdf_response = requests.get(pdf_link, headers=headers, timeout=20)
        pdf_stream = io.BytesIO(pdf_response.content)
        reader = pypdf.PdfReader(pdf_stream)
        
        # 提取前 5 页文本（通常 Predicate Device 都在首页或前几页）
        text_content = ""
        for i in range(min(5, len(reader.pages))):
            extracted = reader.pages[i].extract_text()
            if extracted:
                text_content += extracted + "\n"

        # 第四步：正则提取 K 编号
        # 1. 尝试寻找跟在 "Predicate" 关键词后的 K 编号
        matches = re.findall(r'Predicate.*?(K\s?\d{6})', text_content, re.IGNORECASE | re.DOTALL)
        
        # 2. 如果没找，则抓取所有符合 K+6位数字 格式的内容（排除原设备编号）
        if not matches:
            matches = re.findall(r'K\s?\d{6}', text_content, re.IGNORECASE)
        
        # 清洗：去除中间可能的空格、转大写、去重、排除掉查询编号本身
        clean_results = []
        for m in matches:
            clean_k = m.replace(" ", "").upper()
            if clean_k != current_k:
                clean_results.append(clean_k)
        
        unique_results = sorted(list(set(clean_results)))
        
        return ", ".join(unique_results) if unique_results else "Not Found"

    except requests.exceptions.RequestException:
        return "Network Error"
    except Exception as e:
        return f"Error: {str(e)}"

Основная программа: Пакетная обработка данных Excel
Чтение файлов Excel, построчная обработка K-числ, вызов указанной выше функции обхода, добавление интервалов запросов для предотвращения блокировки IP-адресов и сохранение результатов в новый файл Excel.

In [3]:
# 3. 运行主逻辑
def run_automation():
    print(f"Загрузка файла: {INPUT_FILE}...")
    try:
        # 使用 openpyxl 引擎直接读取 Excel，解决编码问题
        df = pd.read_excel(INPUT_FILE, engine='openpyxl')
    except Exception as e:
        print(f"Не удалось открыть файл.: {e}。Пожалуйста, убедитесь, что имя файла указано правильно и что Excel закрыт.。")
        return

    if COLUMN_NAME not in df.columns:
        print(f"Ошибка: Столбец с именем '{COLUMN_NAME}' не найден в Excel. ")
        return

    total_rows = len(df)
    print(f"Началась автоматическая выгрузка данных, охватывающая в общей сложности {total_rows} устройств")
    
    extracted_data = []
    
    for index, row in df.iterrows():
        sub_num = row[COLUMN_NAME]
        print(f"[{index + 1}/{total_rows}] Обработка: {sub_num}...", end="\r")
        
        # 获取结果
        result = fetch_predicate_device(sub_num)
        extracted_data.append(result)
        
        # 睡眠以防止 IP 被封
        time.sleep(SLEEP_TIME)

    # 将结果存入新列并保存
    df['Extracted Predicate Device'] = extracted_data
    df.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')
    
    print(f"\n\nИзвлечение завершено")
    print(f"Результаты сохранены в: {OUTPUT_FILE}")

if __name__ == "__main__":
    run_automation()

Загрузка файла: data.xlsx...
Началась автоматическая выгрузка данных, охватывающая в общей сложности 72 устройств
[72/72] Обработка: K251071...

Извлечение завершено
Результаты сохранены в: data_with_predicates.xlsx
